In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

def find_downloadables(url, extensions=None, filenames=None):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.content, 'html.parser')
        downloadables = []
        found_urls = set()

        for link in soup.find_all('a', href=True):
            href = link['href']
            if not href or href.startswith('#') or href.startswith('javascript:'):
                continue
            
            absolute_url = urljoin(url, href)
            
            if absolute_url in found_urls:
                continue
            
            parsed_url = urlparse(absolute_url)
            path = parsed_url.path
            filename = path.split('/')[-1] if '/' in path else path

            # Check extensions
            if extensions:
                for ext in extensions:
                    if path.lower().endswith(f'.{ext.lower()}'):
                        found_urls.add(absolute_url)
                        downloadables.append({
                            'filename': filename,
                            'link': absolute_url
                        })
                        break
            
            # Check filenames
            if filenames:
                for target_name in filenames:
                    if filename.lower() == target_name.lower():
                        found_urls.add(absolute_url)
                        downloadables.append({
                            'filename': filename,
                            'link': absolute_url
                        })
                        break

        return downloadables
        
    except Exception as e:
        print(f"Error: {e}")
        return []

In [2]:
from pprint import pprint
from tqdm import tqdm

In [3]:

files = find_downloadables("https://vision.middlebury.edu/stereo/data/scenes2014/zip/", ["zip"])
for file in files:
    print(f"File: {file['filename']}, Link: {file['link']}")

File: Adirondack-imperfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Adirondack-imperfect.zip
File: Adirondack-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Adirondack-perfect.zip
File: Backpack-imperfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Backpack-imperfect.zip
File: Backpack-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Backpack-perfect.zip
File: Bicycle1-imperfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Bicycle1-imperfect.zip
File: Bicycle1-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Bicycle1-perfect.zip
File: Cable-imperfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Cable-imperfect.zip
File: Cable-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Cable-perfect.zip
File: Classroom1-imperfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/z

In [4]:
def remove_keyword(files,keyword):
    
    new_files=[]
    
    for file in files:
        
        link = file['link']
        if not (keyword in link):
            new_files.append(file)
            
    return new_files

files = remove_keyword(files,'imperfect')
for file in files:
    print(f"File: {file['filename']}, Link: {file['link']}")

File: Adirondack-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Adirondack-perfect.zip
File: Backpack-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Backpack-perfect.zip
File: Bicycle1-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Bicycle1-perfect.zip
File: Cable-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Cable-perfect.zip
File: Classroom1-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Classroom1-perfect.zip
File: Couch-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Couch-perfect.zip
File: Flowers-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Flowers-perfect.zip
File: Jadeplant-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Jadeplant-perfect.zip
File: Mask-perfect.zip, Link: https://vision.middlebury.edu/stereo/data/scenes2014/zip/Mask-perfect.zip
File

In [5]:
def generate_org_site(files):
    
    org=[]
    
    for file in files:
        
        link = file['link']
        link = link[:-4]
        direc = link.replace('zip','datasets')
        
        org.append(direc+'/')
        
        org.append(direc+"/ambient/L1/")
        org.append(direc+"/ambient/L2/")
        org.append(direc+"/ambient/L3/")
        org.append(direc+"/ambient/L4/")
        
    return org

org_site = generate_org_site(files)
pprint(org_site)
        

['https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/',
 'https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/ambient/L1/',
 'https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/ambient/L2/',
 'https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/ambient/L3/',
 'https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/ambient/L4/',
 'https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Backpack-perfect/',
 'https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Backpack-perfect/ambient/L1/',
 'https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Backpack-perfect/ambient/L2/',
 'https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Backpack-perfect/ambient/L3/',
 'https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Backpack-perfect/ambient/L4/',
 'https://vision.middlebury.edu/stereo/data/scenes2014/dataset

In [6]:
download_list=[]

default_list = ["im0.png","im1.png","disp0.pfm","disp1.pfm","calib.txt"]
default_list += ["im0e0.png","im0e1.png", "im0e2.png","im0e3.png","im0e4.png",
            "im0e5.png","im0e6.png","im0e7.png","im1e0.png","im1e1.png","im1e2.png","im1e3.png",
            "im1e4.png","im1e5.png","im1e6.png","im1e7.png"
]

for site in tqdm(org_site):
    files = find_downloadables(site, filenames=default_list)
    download_list+=files



  2%|▏         | 2/115 [00:02<02:00,  1.07s/it]

Error: 403 Client Error: Forbidden for url: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/ambient/L1/


  3%|▎         | 3/115 [00:03<02:02,  1.09s/it]

Error: 403 Client Error: Forbidden for url: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/ambient/L2/


  3%|▎         | 4/115 [00:04<01:55,  1.04s/it]

...

 65%|██████▌   | 75/115 [01:20<00:39,  1.01it/s]

Error: 404 Client Error: Not Found for url: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Recycle-perfect/ambient/L4/


 91%|█████████▏| 105/115 [01:49<00:09,  1.06it/s]

Error: 404 Client Error: Not Found for url: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Sword2-perfect/ambient/L4/


100%|██████████| 115/115 [01:59<00:00,  1.04s/it]


In [7]:
for file in download_list:
    print(f"File: {file['filename']}, Link: {file['link']}")

File: calib.txt, Link: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/calib.txt
File: disp0.pfm, Link: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/disp0.pfm
File: disp1.pfm, Link: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/disp1.pfm
File: im0.png, Link: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/im0.png
File: im1.png, Link: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Adirondack-perfect/im1.png
File: calib.txt, Link: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Backpack-perfect/calib.txt
File: disp0.pfm, Link: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Backpack-perfect/disp0.pfm
File: disp1.pfm, Link: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Backpack-perfect/disp1.pfm
File: im0.png, Link: https://vision.middlebury.edu/stereo/data/scenes2014/datasets/Backpack-perfect/im

In [8]:
import requests
import os
from urllib.parse import unquote, urlparse

def download_files(file_list, download_location, org_site):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    downloaded_files = []
    
    for file_info in file_list:
        try:
            file_url = file_info['link']
            
            # Parse both URLs
            parsed_file_url = urlparse(file_url)
            parsed_org_site = urlparse(org_site)
            
            # Extract the relative path from org_site
            file_path = parsed_file_url.path
            org_path = parsed_org_site.path
            
            # Remove the org_site path from the file path to get relative path
            if file_path.startswith(org_path):
                relative_path = file_path[len(org_path):].lstrip('/')
            else:
                relative_path = file_path.lstrip('/')
            
            # Construct full download path
            full_download_path = os.path.join(download_location, relative_path)
            
            # Check if file already exists
            if os.path.exists(full_download_path):
                print(f"✓ File already exists: {relative_path}")
                downloaded_files.append({
                    'filename': os.path.basename(relative_path),
                    'path': full_download_path,
                    'status': 'exists'
                })
                continue
            
            # Create directory structure if it doesn't exist
            file_dir = os.path.dirname(full_download_path)
            if file_dir and not os.path.exists(file_dir):
                os.makedirs(file_dir)
            
            print(f"Downloading: {relative_path}")
            
            response = requests.get(file_url, headers=headers, stream=True, timeout=30)
            response.raise_for_status()
            
            with open(full_download_path, 'wb') as file:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        file.write(chunk)
            
            downloaded_files.append({
                'filename': os.path.basename(relative_path),
                'path': full_download_path,
                'status': 'success'
            })
            print(f"✓ Downloaded: {relative_path}")
            
        except Exception as e:
            print(f"✗ Failed to download {file_info['filename']}: {e}")
            downloaded_files.append({
                'filename': file_info['filename'],
                'status': 'failed',
                'error': str(e)
            })
    
    return downloaded_files

In [9]:


downloaded = download_files(download_list, '/mnt/Velocity_Vault/Project_Storage/raw_dataset/',
                            "https://vision.middlebury.edu/stereo/data/scenes2014/datasets/")

print(f"Successfully downloaded {len([f for f in downloaded if f['status'] == 'success'])} files")

✓ File already exists: Adirondack-perfect/calib.txt
✓ File already exists: Adirondack-perfect/disp0.pfm
✓ File already exists: Adirondack-perfect/disp1.pfm
✓ File already exists: Adirondack-perfect/im0.png
✓ File already exists: Adirondack-perfect/im1.png
✓ File already exists: Backpack-perfect/calib.txt
✓ File already exists: Backpack-perfect/disp0.pfm
✓ File already exists: Backpack-perfect/disp1.pfm
✓ File already exists: Backpack-perfect/im0.png
✓ File already exists: Backpack-perfect/im1.png
✓ File already exists: Backpack-perfect/ambient/L1/im0e0.png
✓ File already exists: Backpack-perfect/ambient/L1/im0e1.png
✓ File already exists: Backpack-perfect/ambient/L1/im0e2.png
✓ File already exists: Backpack-perfect/ambient/L1/im0e3.png
✓ File already exists: Backpack-perfect/ambient/L1/im0e4.png
✓ File already exists: Backpack-perfect/ambient/L1/im0e5.png
✓ File already exists: Backpack-perfect/ambient/L1/im0e6.png
✓ File already exists: Backpack-perfect/ambient/L1/im0e7.png
✓ File alr